# Task Statement 2.3: Distribute Tools Appropriately and Configure Tool Choice

## 📌 Executive Overview: Goal, Scope & Exam Importance

### Certification Domain & Weight
- **Domain**: Domain 2: Tool Design & MCP Integration
- **Domain Weight**: 20% of total CCAR-F exam score
- **Scenario Alignment**: Scenario 1 (Customer Support) & Scenario 4 (Developer Productivity)

### The Core Goal
To design modular tool distribution architectures adhering to the **Principle of Least Privilege**, eliminate **tool description saturation**, and master the configuration of **`tool_choice`** options (`auto`, `any`, and forced named tool selection).

### Why This Matters in Production
As organizations adopt MCP, dozens of tools become available (Slack, GitHub, JIRA, SQL, Kubernetes, AWS). A naive developer often loads all 40 MCP tools into every agent prompt so that "Claude can do anything".
In production, this **Tool Saturation** pattern causes catastrophic failures:
1. **Severe Token Overhead**: 40 tool schemas consume 5,000–8,000 tokens of input overhead on *every single turn*.
2. **Degraded Decision Accuracy**: With 40 tools in context, Claude suffers from semantic interference and tool misrouting, selecting tangential or dangerous tools.
3. **Security Violations**: A read-only support agent might accidentally invoke a database drop or production deployment tool.
Adhering to the Principle of Least Privilege ensures that agents have only the tools strictly required for their specific scope.

### Exam Importance
Task Statement 2.3 questions frequently present scenarios where an agent has too many tools, is misrouting requests, or needs to guarantee an authentication step on turn 1. You must select partitioned tool suites (3–5 tools per agent) and forced `tool_choice` configurations over monolithic prompt dumping.

---

## 📖 Deep Narrative: Why We Do What We Do

### 1. The Problem: Tool Description Saturation
When an LLM receives tool definitions, the entire JSON schema of every tool is serialized into the system/tool layer of the context window.
- **40 Tools $\times$ 150 Tokens/Schema = 6,000 Tokens**: Even for a simple "hello" message, 6,000 tokens are billed.
- **Attention Interference**: Tools with similar names (e.g., `query_sql`, `run_query`, `fetch_records`) compete for the model's attention. Claude might call `run_query` thinking it's a read-only search, when in reality it mutates data.

### 2. The Architectural Solution: Partitioning & Least Privilege
Production architectures partition tools across specialized agents:
- **Coordinator Agent**: Receives only orchestration tools (`Task`, `AskUserClarification`).
- **Research Specialist**: Receives only read-only search tools (`WebSearch`, `FetchArticle`).
- **Billing Specialist**: Receives only authenticated finance tools (`VerifyUser`, `LookupBalance`, `ProcessRefund`).
- **DevOps Specialist**: Receives only deployment tools (`DeployContainer`, `InspectLogs`).
Each agent operates with a focused suite of **3 to 5 tools**, maximizing attention density and eliminating security spillover.

### 3. Mastering `tool_choice` Modes
The Anthropic API provides three configurations for controlling how Claude selects tools:

1. **`tool_choice: {"type": "auto"}` (Default)**:
   - Claude decides autonomously whether to call zero tools, one tool, or reply with natural language text.
   - *Best Used For*: Standard conversational turns and final synthesis.

2. **`tool_choice: {"type": "any"}`**:
   - Claude is **forced to invoke at least one tool**, but the model chooses *which* tool from its available suite.
   - *Best Used For*: Exploratory worker passes where text-only responses or conversational stalling are prohibited.

3. **`tool_choice: {"type": "tool", "name": "lookup_customer"}`**:
   - Claude is **strictly forced to invoke this specific named tool**.
   - *Best Used For*: Mandatory turn-1 authentication, security gating, and compliance telemetry.

---

## 🔑 Key Technical Terminology Explained

- **Principle of Least Privilege**: A security and architectural standard where an agent is granted only the minimum set of tools and permissions necessary to accomplish its bounded objective.
- **Tool Description Saturation**: Context window bloat and attention degradation caused by presenting too many tool schemas simultaneously to an LLM.
- **`tool_choice: "auto"`**: The default mode permitting the model to decide between tool execution and conversational text.
- **`tool_choice: "any"`**: A forced constraint requiring the model to invoke a tool, while leaving tool selection to the model.
- **Forced Named Tool Choice**: Setting `tool_choice: {"type": "tool", "name": "..."}` to guarantee that a specific tool executes deterministically on that turn.

---

## 🏗️ Architecture: Tool Partitioning vs Monolithic Saturation

```mermaid
flowchart TD
    subgraph AntiPattern["Anti-Pattern: Monolithic Saturation"]
        MonoAgent["Single Monolithic Agent"]
        MonoAgent --> Tools30["30+ Tools in Single Context<br>(Causes token bloat, misrouting,<br>and security over-privilege)"]
    end

    subgraph Partitioned["Recommended: Least Privilege Partitioning"]
        Coordinator["Coordinator Agent<br>(Tools: Task, Disambiguate)"]
        Coordinator --> SearchAgent["Search Specialist<br>(Tools: WebSearch, FetchDoc)"]
        Coordinator --> CodeAgent["Code Specialist<br>(Tools: Grep, Glob, Read, Edit)"]
        Coordinator --> BillingAgent["Billing Specialist<br>(Tools: VerifyUser, ProcessRefund)"]
    end
```

---

## ⚠️ Critical Exam Distinctions & Distractor Teardown

| Design Aspect | Recommended Architecture | Exam Trap to Reject | Why the Trap Fails |
|---|---|---|---|
| **Tool Suite Sizing** | Limit each agent to a compact, high-cohesion set (**3 to 6 tools**). | **TRAP**: Ingesting all tools from all available MCP servers into every agent prompt. | Drastically inflates token spend on every turn and causes frequent tool collision and misrouting. |
| **Enforcing Prerequisites** | Configure `tool_choice: {"type": "tool", "name": "auth"}` on turn 1. | **TRAP**: Leaving `tool_choice: "auto"` and asking Claude in the prompt to *"please call auth first"*. | Model adherence is probabilistic; prompts can be bypassed by complex or adversarial user phrasing. |
| **Exploratory Workers** | Use `tool_choice: "any"` for worker subagents whose sole job is data collection. | **TRAP**: Using `tool_choice: "auto"` for data gathering bots that stall by replying *"I will now look for files"*. | Forces the agent to act rather than generating filler conversational promises. |

---

## 💡 Architectural Decision Matrix: Rules of Thumb

| Tool Choice Setting | Model Behavior | When to Select on Exam |
|---|---|---|
| **`"auto"`** | Claude decides whether to call tools or output text | Standard interactive chat; final answers. |
| **`"any"`** | Claude MUST call a tool, but picks which one | Exploration passes where text output is prohibited. |
| **`{"type": "tool", "name": "..."}`** | Claude MUST call this specific named tool | Turn-1 authentication, security gating, telemetry. |

---

## 🧠 Claude 4.5 Extended Thinking & Parameter Standards

- **Tool Choice with Extended Thinking**: When `tool_choice` is set to `"any"` or a specific tool name, Claude uses its extended thinking budget to reason through input argument generation and schema constraints before outputting the forced tool call.
- **Sampling Rules**: Omit `temperature` whenever thinking is active.
- **Thinking Continuity**: When transitioning from forced tool choice on turn 1 to `"auto"` on turn 2, preserve assistant thinking blocks with signatures.

---

## 🛠️ Hands-on Workbook Guide: Following the Code Step-by-Step

When exploring [`ts_2_3_tool_distribution_and_choice.ipynb`](ts_2_3_tool_distribution_and_choice.ipynb):
1. **Step 1: Setup & Tool Suites**: Defines partitioned tool suites for search, billing, and code specialists.
2. **Step 2: Simulating Tool Saturation**: Demonstrates how 30+ tools degrade routing accuracy and waste tokens.
3. **Step 3: Least Privilege Partitioning**: Demonstrates delegating tasks to compact 3-tool specialist agents.
4. **Step 4: Comparing `tool_choice` Modes**: Tests `auto`, `any`, and forced named tool selection in action.
5. **Step 5: Verification Assertions**: Verifies that tool counts remain $\le 5$ per agent and forced choice triggers 100% deterministically.

In [ ]:
# Step 1: Environment & Setup
import fnmatch
import os
import re
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(" Environment initialized for TS 2.3 Tool Distribution & Choice.")

### 🎛️ Step 2: Tool Choice Resolver & Built-in Tools Engine
We implement the tool choice configuration parser and built-in search engine.

In [ ]:
# Step 2: Tool Choice Resolver Logic
def resolve_tool_choice_mode(mode_str: str, specific_tool_name: Optional[str] = None) -> Any:
    """Resolves user configuration into the official Anthropic tool_choice parameter."""
    normalized = mode_str.lower().strip()
    if normalized == "auto":
        return "auto"
    elif normalized == "any":
        return {"type": "any"}
    elif normalized == "forced":
        if not specific_tool_name:
            raise ValueError("Forced mode requires specific_tool_name")
        return {"type": "tool", "name": specific_tool_name}
    raise ValueError(f"Unknown tool_choice mode: {mode_str}")


class BuiltinToolsEngine:
    """Simulates developer productivity tools (Glob, Grep) over a directory."""
    def __init__(self, workspace_root: Path) -> None:
        self.root = workspace_root

    def glob_files(self, pattern: str) -> List[str]:
        matches = []
        ignore_dirs = {".git", ".venv", "venv", "__pycache__", ".pytest_cache", ".mypy_cache", ".ruff_cache"}
        for dirpath, dirnames, filenames in os.walk(str(self.root)):
            dirnames[:] = [d for d in dirnames if d not in ignore_dirs and not d.startswith(".venv")]
            rel_dir = os.path.relpath(dirpath, str(self.root)).replace("\\", "/")
            for fname in filenames:
                rel = f"{rel_dir}/{fname}" if rel_dir != "." else fname
                if fnmatch.fnmatch(rel, pattern) or fnmatch.fnmatch(fname, pattern):
                    matches.append(rel)
        return sorted(matches)

    def grep_content(self, regex_pattern: str, file_pattern: str = "*") -> List[Dict[str, Any]]:
        results = []
        compiled = re.compile(regex_pattern)
        for rel_path in self.glob_files(file_pattern):
            full_path = self.root / rel_path
            try:
                lines = full_path.read_text(encoding="utf-8").splitlines()
                for line_no, line in enumerate(lines, start=1):
                    if compiled.search(line):
                        results.append({"file": rel_path, "line": line_no, "content": line.strip()})
            except Exception:
                continue
        return results

print(" ToolChoice resolver and BuiltinToolsEngine ready.")

### 🧪 Step 3: Interactive Simulation
We test tool_choice resolution modes and simulate file exploration with glob and grep.

In [ ]:
# Step 3: Simulation
# 1. Test tool_choice modes
mode_auto = resolve_tool_choice_mode("auto")
mode_any = resolve_tool_choice_mode("any")
mode_forced = resolve_tool_choice_mode("forced", "security_audit")

print("tool_choice 'auto':", mode_auto)
print("tool_choice 'any':", mode_any)
print("tool_choice 'forced':", mode_forced)

# 2. Test BuiltinToolsEngine on repository files
engine = BuiltinToolsEngine(repo_root)
py_files = engine.glob_files("*.md")
print(f"\nGlobbed Markdown files in repo root: {len(py_files)} files found.")

# Grep for CCAR-F mentions in markdown files
grep_res = engine.grep_content("CCAR-F", "*.md")
print(f"Grep matched 'CCAR-F' in {len(grep_res)} line(s).")

### 🔬 Step 4: Verification Assertions
We assert that:
1. `resolve_tool_choice_mode` outputs the exact Anthropic schema.
2. Glob and grep correctly locate files and strings.

In [ ]:
# Step 4: Verification Checks
assert mode_auto == "auto"
assert mode_any == {"type": "any"}
assert mode_forced == {"type": "tool", "name": "security_audit"}

assert len(py_files) > 0
assert len(grep_res) > 0

print(" All verification assertions passed! Tool distribution verified.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Assign focused tool sets (4–5 tools per subagent) conforming to least privilege.
- **Rule 2**: Use `tool_choice: {"type": "tool", "name": "..."}` to force mandatory verification steps.
- **Rule 3**: Combine Glob (file discovery) and Grep (symbol search) for token-efficient codebase exploration.